# MODIS to VIIRS harmonization, all of Algeria: the data decisions, tested

What this notebook does, in order. Every section prints a table and draws a figure, so each decision can be judged with numbers.

| Section | Question |
|---|---|
| 1 | Load the two archives for the whole country, with every detection type kept |
| A | Which detection types exist, and does their share change over the years? (why keep type 0 only) |
| B | When did each satellite start? (why drop 2000-2002) |
| C | Which months look like outages, by a rule that does not use our hand-made list? |
| D | How much did detections change from 2018-2021 to 2022-2025 for MODIS and VIIRS, and is the cause visible? |
| E | At what time of day did each satellite pass over Algeria, year by year? |
| F | Does the VIIRS / MODIS ratio drift, and does the type choice change it? |
| G | Build the block-month table (0.5 degree blocks) used by the models |
| H | The four translation models, each in its own block: straight curve, curved fit, gradient boosting, neural network |
| I | Compare the data decisions: type 0 vs all types, outages removed vs kept, training with or without the drift years |
| J | How stable is the neural network across random seeds? |
| K | Preview of the reconstruction 2000-2012 and the one-satellite problem |

How to run: set `RAWD` below (or the environment variable `FIRE_RAW`), then Run All. The raw archives must be the whole-country FIRMS files `modis_archive.csv` and `viirs_archive.csv`. Figures are saved to `FIGDIR` and the result tables to CSV there too.

Nothing in this notebook has been run on your real files. It was run end to end on made-up archives that contain a planted drop, planted outages and planted static sources, so the code works, but its numbers on your data are unknown until you run it.

In [2]:
import os
import calendar
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ------------------------------ settings ------------------------------
RAWD = Path(os.environ.get("FIRE_RAW", "../data/raw/algeria"))     # folder with modis_archive.csv and viirs_archive.csv
FIGDIR = Path(os.environ.get("FIRE_FIG", "../figures"))            # figures and result tables are saved here
FIGDIR.mkdir(parents=True, exist_ok=True)
CHUNK = 500_000
BLOCK = 0.5                                   # block size in degrees (about 50 km)
NORTH_LAT = 34.0                              # north = latitude >= 34.0, south = below
REF_YEARS, TEST_YEARS = (2018, 2021), (2022, 2025)      # periods compared in section D
HIDDEN_YEARS = (2013, 2015)                   # years used as the hidden test in section I
YOUR_OUTAGE_MONTHS = ["2017-12", "2018-10", "2022-04", "2022-07", "2022-08", "2024-01", "2024-02", "2024-06",
                      "2024-07", "2024-11", "2025-12", "2026-01", "2026-03", "2026-04", "2026-07"]
MIN_MISSING_DAYS = {"viirs": 1, "modis": 10}  # outage rule 1: a month with at least this many days without ANY detection (MODIS sees few static sources, so it needs a looser limit)
STATIC_RATIO_LIMIT = 0.5                      # outage rule 2: static-source detections below 50% of their normal level
LAST_FULL_YEAR = 2025                         # the analysis stops at the end of this year
T0 = time.time()

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "figure.figsize": (11, 4.2)})
RESULTS = {}                                  # every table is stored here and saved to CSV at the end

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGDIR / f"{name}.png", dpi=130)
    plt.show()

def keep(name, table):
    RESULTS[name] = table
    return table
print("raw folder:", RAWD.resolve())

raw folder: C:\Everything\space-apps-fire\data\raw\algeria


## 1. Load both archives, whole country, every detection type kept

We keep all rows here, and filter later, so that the same loaded data can be used for the type 0 version and the all types version.
Each row gets: the day, the month (`ym`), the 0.01 degree cell (about 1 km, `cy`, `cx`), the 0.5 degree block (`by`, `bx`), the type code, the satellite name, the UTC hour, and a flag for nominal confidence or better (MODIS at least 30, VIIRS n or h).

In [3]:
def load(kind):
    path = RAWD / f"{kind}_archive.csv"
    header = pd.read_csv(path, nrows=0).columns.tolist()
    want = ["latitude", "longitude", "acq_date", "acq_time", "confidence", "type", "satellite"]
    use = [c for c in want if c in header]
    print(f"{kind}: columns used {use} | not in file {[c for c in want if c not in header]}")
    parts = []
    for ch in pd.read_csv(path, usecols=use, chunksize=CHUNK):
        out = pd.DataFrame({"lat": ch["latitude"].astype("float32"), "lon": ch["longitude"].astype("float32")})
        out["date"] = pd.to_datetime(ch["acq_date"])
        out["year"] = out.date.dt.year.astype("int16")
        out["month"] = out.date.dt.month.astype("int8")
        out["ym"] = out.date.dt.strftime("%Y-%m")
        out["cy"] = np.floor(ch["latitude"] / 0.01).astype("int32")
        out["cx"] = np.floor(ch["longitude"] / 0.01).astype("int32")
        out["by"] = np.floor(ch["latitude"] / BLOCK).astype("int32")
        out["bx"] = np.floor(ch["longitude"] / BLOCK).astype("int32")
        out["type"] = pd.to_numeric(ch["type"], errors="coerce").astype("float32") if "type" in ch else np.float32(np.nan)
        out["sat"] = ch["satellite"].astype(str) if "satellite" in ch else "unknown"
        t = pd.to_numeric(ch["acq_time"], errors="coerce") if "acq_time" in ch else pd.Series(np.nan, index=ch.index)
        out["hour"] = ((t // 100) + (t % 100) / 60).astype("float32")           # UTC decimal hour
        if kind == "viirs":
            out["conf_ok"] = ch["confidence"].astype(str).isin(["n", "h"])
        else:
            out["conf_ok"] = pd.to_numeric(ch["confidence"], errors="coerce") >= 30
        parts.append(out)
    df = pd.concat(parts, ignore_index=True)
    df = df[df.year <= LAST_FULL_YEAR]
    print(f"{kind}: {len(df):,} rows | {df.date.min().date()} to {df.date.max().date()}")
    return df

D = {"modis": load("modis"), "viirs": load("viirs")}
print("loaded in", round((time.time() - T0) / 60, 1), "min")

modis: columns used ['latitude', 'longitude', 'acq_date', 'acq_time', 'confidence', 'type', 'satellite'] | not in file []
modis: 225,466 rows | 2000-11-01 to 2025-12-31
viirs: columns used ['latitude', 'longitude', 'acq_date', 'acq_time', 'confidence', 'type', 'satellite'] | not in file []
viirs: 1,325,643 rows | 2012-07-04 to 2025-12-31
loaded in 0.4 min


## A. Detection types: what is in the files, and does it change over time?

FIRMS labels each detection: 0 = presumed vegetation fire, other codes = static land sources such as gas flares, volcanoes, or offshore detections (confirm the exact codes in the FIRMS readme that comes with your download).

Why this matters for the model: a static source fires every day, so it adds a constant offset that has nothing to do with burning activity, and the two sensors do not see these sources equally well.
Two things to look for in the figure: a large non-zero share (what we would be mixing in) and a sudden change of share in some year (a change in how the type was assigned, which would be a bias of its own).

In [ ]:
def type_share(df):
    d = df[df.conf_ok]
    label = d.type.fillna(-1).astype(int)
    counts = pd.crosstab(d.year, label)
    share = 100 * counts.div(counts.sum(axis=1), axis=0)
    share.columns = ["unlabelled" if c == -1 else f"type {int(c)}" for c in share.columns]
    return counts, share

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, kind in zip(axes, ["modis", "viirs"]):
    counts, share = type_share(D[kind])
    keep(f"A_type_share_{kind}", share.round(1))
    share.plot(kind="bar", stacked=True, ax=ax, width=0.85)
    ax.set_title(f"{kind.upper()}: share of detections by type (%)")
    ax.set_xlabel("year"); ax.set_ylabel("% of rows")
    ax.legend(loc="lower left", fontsize=8)
savefig("A_type_share")

for kind in ("modis", "viirs"):
    d = D[kind][D[kind].conf_ok & (D[kind].year >= 2013)]
    n_non0 = int(((d.type != 0) & d.type.notna()).sum())
    print(f"{kind.upper()} 2013-{LAST_FULL_YEAR}: {len(d):,} rows | not type 0 and labelled: {n_non0:,} ({100 * n_non0 / len(d):.1f}%) | unlabelled: {int(d.type.isna().sum()):,}")
display(keep("A_type_share_viirs_table", type_share(D["viirs"])[1].round(1)))

## B. Satellites: when did each one start?

The claim to test: before Aqua flew, MODIS had only one satellite (Terra), so it saw about half as many overpasses, and its counts are not comparable to later years.
The first dates below come from the archive itself, not from memory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, kind in zip(axes, ["modis", "viirs"]):
    d = D[kind]
    table = pd.crosstab(d.year, d.sat)
    table.plot(ax=ax, marker="o")
    ax.set_yscale("log")
    ax.set_title(f"{kind.upper()}: rows per year by satellite (log scale)")
    ax.set_xlabel("year"); ax.set_ylabel("rows")
    first_last = d.groupby("sat").date.agg(["min", "max"])
    first_last["min"], first_last["max"] = first_last["min"].dt.strftime("%Y-%m-%d"), first_last["max"].dt.strftime("%Y-%m-%d")
    print(f"\n{kind.upper()}: first and last date per satellite")
    display(first_last.rename(columns={"min": "first date", "max": "last date"}))
savefig("B_satellites")

modis_all = D["modis"][D["modis"].conf_ok & (D["modis"].type == 0)]
per_year = modis_all.drop_duplicates(["cy", "cx", "date", "sat"]).groupby(["year", "sat"]).size().unstack(fill_value=0)
per_year["Terra+Aqua"] = per_year.sum(axis=1)
display(keep("B_modis_cellday_per_year_by_satellite", per_year))

## C. Outage months: found by a rule, then compared with your list

Idea: a satellite that works detects something somewhere in Algeria almost every day, because static sources such as gas flares burn constantly. So an outage looks like (rule 1) calendar days with no detection at all, or (rule 2) a month where the static-source detections fall far below their normal level, while fire detections alone could be low simply because it was a quiet month.
Rule 2 only runs for months where almost all rows carry a type label (the most recent rows may have none).
The thresholds are my first choice and are not tuned. Read the table before trusting them.

In [ ]:
def month_table(kind, start):
    d = D[kind]
    last_ym = d.ym.max()                                                  # the last month may be partial and is left out
    months = [str(p) for p in pd.period_range(start, pd.Period(last_ym, "M") - 1, freq="M")]
    g = d[d.ym.isin(months)].groupby("ym")
    t = pd.DataFrame(index=months)
    t["days in month"] = [calendar.monthrange(int(m[:4]), int(m[5:7]))[1] for m in months]
    t["days with a detection"] = g.date.nunique().reindex(months).fillna(0).astype(int)
    t["days missing"] = t["days in month"] - t["days with a detection"]
    t["labelled share"] = g.type.apply(lambda s: s.notna().mean()).reindex(months)
    static = d[d.ym.isin(months) & d.type.notna() & (d.type != 0)].groupby("ym").size().reindex(months).fillna(0)
    t["static detections"] = static
    normal = static.rolling(13, center=True, min_periods=7).median().replace(0, np.nan)
    t["static / normal"] = static / normal
    rule1 = t["days missing"] >= MIN_MISSING_DAYS[kind]
    rule2 = (t["static / normal"] < STATIC_RATIO_LIMIT) & (t["labelled share"] >= 0.9)
    t["flagged by"] = np.where(rule1 & rule2, "both", np.where(rule1, "missing days", np.where(rule2, "static drop", "")))
    t["on your list"] = t.index.isin(YOUR_OUTAGE_MONTHS)
    return t

OUT = {}
for kind, start in (("viirs", "2013-01"), ("modis", "2003-01")):
    OUT[kind] = month_table(kind, start)
    flagged = OUT[kind][OUT[kind]["flagged by"] != ""]
    print(f"\n{kind.upper()}: {len(flagged)} months flagged out of {len(OUT[kind])} checked")
    display(keep(f"C_flagged_{kind}", flagged.round(2)))
    inside = [m for m in YOUR_OUTAGE_MONTHS if m in OUT[kind].index]
    print("  your list inside the checked period:", len(inside))
    print("  on your list and flagged by the rule:", [m for m in inside if OUT[kind].loc[m, "flagged by"] != ""])
    print("  on your list but NOT flagged:        ", [m for m in inside if OUT[kind].loc[m, "flagged by"] == ""])
    print("  flagged but NOT on your list:        ", [m for m in flagged.index if m not in YOUR_OUTAGE_MONTHS])

DETECTED = set(OUT["viirs"].index[OUT["viirs"]["flagged by"] != ""])
OUTAGE_USER = set(YOUR_OUTAGE_MONTHS)
OUTAGE_ALL = OUTAGE_USER | DETECTED                    # months removed from scoring and, in some scenarios, from training

t = OUT["viirs"]
x = np.arange(len(t))
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].bar(x, t["days missing"], color="tab:red")
axes[0].set_ylabel("days with no VIIRS detection"); axes[0].set_title("VIIRS: calendar days without any detection, per month")
axes[1].plot(x, t["static / normal"], marker=".")
axes[1].axhline(STATIC_RATIO_LIMIT, color="k", ls="--", label=f"limit {STATIC_RATIO_LIMIT}")
axes[1].set_ylabel("static detections / normal"); axes[1].set_title("VIIRS: static-source detections relative to their 13-month median")
for ax in axes:
    for i, m in enumerate(t.index):
        if m in OUTAGE_USER:
            ax.axvline(i, color="tab:orange", alpha=0.5, lw=1)
step = max(1, len(t) // 20)
axes[1].set_xticks(x[::step]); axes[1].set_xticklabels(t.index[::step], rotation=60, fontsize=8)
axes[1].legend()
savefig("C_outages")
print("orange lines = months on your list")

## D. The 2022-2025 drop: MODIS against VIIRS

The claim to test: from 2018-2021 to 2022-2025 detections fell by about 49% for MODIS and about 9% for VIIRS.
We use fire cell-days (one 1 km cell counted once per day), compare the same calendar months in both periods so that the season mix cannot fake a change, and show the result with and without the outage months removed. The same is repeated for all detection types, for Terra and Aqua separately, and for the north and south of the country.

Why this matters for the model: if MODIS counts fell for reasons that are not about fire (a change in the satellites' overpass time, for example), then a model trained on those years learns a relationship that does not describe 2003-2012.

In [ ]:
MONTHS_LONG = [str(p) for p in pd.period_range("2000-01", f"{LAST_FULL_YEAR}-12", freq="M")]
ZONES = {"all": lambda d: d, "north": lambda d: d[d.lat >= NORTH_LAT], "south": lambda d: d[d.lat < NORTH_LAT]}
GROUP_DEF = {"MODIS Terra+Aqua": ("modis", None), "MODIS Terra": ("modis", "Terra"),
             "MODIS Aqua": ("modis", "Aqua"), "VIIRS": ("viirs", None)}

def select(kind, scope):
    d = D[kind][D[kind].conf_ok]
    if scope == "type0":
        d = d[d.type == 0]
    return d

def monthly_series(d):
    d = d.drop_duplicates(["cy", "cx", "date"])                       # fire cell-days
    return d.groupby("ym").size().reindex(MONTHS_LONG).fillna(0)

SERIES = {}
for scope in ("type0", "all"):
    for group, (kind, sat) in GROUP_DEF.items():
        d = select(kind, scope)
        if sat:
            d = d[d.sat.str.contains(sat, case=False)]
        for zone, fn in ZONES.items():
            SERIES[(scope, group, zone)] = monthly_series(fn(d))
print("monthly series built:", len(SERIES), "| minutes so far:", round((time.time() - T0) / 60, 1))

def change(series, bad_months):
    s = series.drop(labels=[m for m in bad_months if m in series.index])
    df = pd.DataFrame({"n": s})
    df["year"], df["month"] = df.index.str[:4].astype(int), df.index.str[5:7].astype(int)
    ref = df[(df.year >= REF_YEARS[0]) & (df.year <= REF_YEARS[1])].groupby("month").n.mean()
    tst = df[(df.year >= TEST_YEARS[0]) & (df.year <= TEST_YEARS[1])].groupby("month").n.mean()
    common = ref.index.intersection(tst.index)                          # the same calendar months in both periods
    return 100 * (tst[common].sum() / ref[common].sum() - 1)

rows = []
for (scope, group, zone), s in SERIES.items():
    rows.append({"types": "type 0 only" if scope == "type0" else "all types", "group": group, "zone": zone,
                 "change % (no month removed)": change(s, set()),
                 "change % (outage months removed)": change(s, OUTAGE_ALL)})
CHANGE = pd.DataFrame(rows)
display(keep("D_change_2022_2025_vs_2018_2021", CHANGE.round(1)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
groups = list(GROUP_DEF)
for ax, zone in zip(axes, ZONES):
    sub = CHANGE[CHANGE.zone == zone].pivot(index="group", columns="types", values="change % (outage months removed)").reindex(groups)
    sub.plot(kind="bar", ax=ax, width=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"zone: {zone}"); ax.set_xlabel(""); ax.set_ylabel("change in fire cell-days (%)")
    ax.tick_params(axis="x", rotation=25)
fig.suptitle(f"{TEST_YEARS[0]}-{TEST_YEARS[1]} versus {REF_YEARS[0]}-{REF_YEARS[1]}, outage months removed, same calendar months")
savefig("D_change_by_group")

# the monthly series themselves, scaled so that the reference period averages 100
x = pd.to_datetime([m + "-01" for m in MONTHS_LONG])
fig, ax = plt.subplots(figsize=(14, 4.5))
for group, colour in (("MODIS Terra+Aqua", "tab:blue"), ("VIIRS", "tab:red")):
    s = SERIES[("type0", group, "all")].copy()
    s[s.index.isin(OUTAGE_ALL)] = np.nan
    ref_mean = s[[m for m in s.index if REF_YEARS[0] <= int(m[:4]) <= REF_YEARS[1]]].mean()
    s = 100 * s / ref_mean
    s.index = x
    s = s[s.index >= "2013-01-01"]
    ax.plot(s.index, s.values, color=colour, alpha=0.25, lw=1)
    ax.plot(s.index, s.rolling(12, min_periods=8).mean().values, color=colour, lw=2.2, label=f"{group} (12-month mean)")
ax.axvspan(pd.Timestamp(f"{TEST_YEARS[0]}-01-01"), pd.Timestamp(f"{TEST_YEARS[1]}-12-31"), color="grey", alpha=0.12)
ax.axhline(100, color="k", lw=0.8)
ax.set_title("Type 0 fire cell-days per month, outage months blank, reference period = 100"); ax.set_ylabel("index"); ax.legend()
savefig("D_monthly_index")

## E. Time of day: did the overpass change?

Satellites fly at fixed local times. If MODIS' passing time moved while VIIRS' did not, MODIS sees the daily fire cycle at a different moment, and its counts can change with no change in the fires.
Below: the median acquisition time (UTC, decimal hours, 13.5 = 13:30) of type 0 detections, per year. Algeria is within about one hour of UTC.

In [ ]:
hour_groups = {"MODIS Terra": ("modis", "Terra"), "MODIS Aqua": ("modis", "Aqua"), "VIIRS": ("viirs", None)}
hours = {}
for group, (kind, sat) in hour_groups.items():
    d = select(kind, "type0")
    if sat:
        d = d[d.sat.str.contains(sat, case=False)]
    d = d[(d.year >= 2013) & (d.year <= LAST_FULL_YEAR)]
    hours[group] = d.groupby("year").hour.median()
HOURS = pd.DataFrame(hours)
display(keep("E_median_utc_hour_per_year", HOURS.round(2)))

fig, ax = plt.subplots(figsize=(11, 4.2))
for group in HOURS:
    ax.plot(HOURS.index, HOURS[group], marker="o", label=group)
ax.axvspan(TEST_YEARS[0] - 0.5, TEST_YEARS[1] + 0.5, color="grey", alpha=0.12)
ax.set_title("Median acquisition time of type 0 detections (UTC hour)"); ax.set_xlabel("year"); ax.set_ylabel("hour"); ax.legend()
savefig("E_overpass_hour")

## F. Does the VIIRS / MODIS relationship drift, and does the type choice change it?

For every year: total VIIRS fire cell-days divided by total MODIS (Terra+Aqua) fire cell-days, outage months removed.
A flat line means the two sensors keep the same proportion. A line that bends means one curve cannot translate all years, whatever model we use.

In [ ]:
ratios = {}
for scope in ("type0", "all"):
    for zone in ZONES:
        v = SERIES[(scope, "VIIRS", zone)].drop(labels=[m for m in OUTAGE_ALL if m in MONTHS_LONG])
        m = SERIES[(scope, "MODIS Terra+Aqua", zone)].drop(labels=[m for m in OUTAGE_ALL if m in MONTHS_LONG])
        v, m = v[[k for k in v.index if int(k[:4]) >= 2013]], m[[k for k in m.index if int(k[:4]) >= 2013]]
        ratios[(scope, zone)] = v.groupby(lambda k: int(k[:4])).sum() / m.groupby(lambda k: int(k[:4])).sum()
RATIO = pd.DataFrame(ratios)
display(keep("F_viirs_over_modis_per_year", RATIO.round(2)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
for ax, zone in zip(axes, ZONES):
    ax.plot(RATIO.index, RATIO[("type0", zone)], marker="o", label="type 0 only")
    ax.plot(RATIO.index, RATIO[("all", zone)], marker="s", label="all types")
    ax.axvspan(TEST_YEARS[0] - 0.5, TEST_YEARS[1] + 0.5, color="grey", alpha=0.12)
    ax.set_title(f"zone: {zone}"); ax.set_xlabel("year"); ax.set_ylabel("VIIRS / MODIS cell-days"); ax.legend()
savefig("F_ratio_per_year")

## G. The block-month table used by the models

Unit: one 0.5 degree block in one month. Targets and inputs are fire cell-days (not rows), so the 375 m pixel size of VIIRS does not enter the model; a separate factor can convert to rows at the very end.

- target `v`: VIIRS cell-days in the block-month (known from 2013-01; unknown before)
- inputs: `lm_all`, `lm_terra`, `lm_aqua` = log(1 + MODIS cell-days, all / Terra only / Aqua only), the block centre (`blat`, `blon`), and the month as a sine and cosine
- rows: every block that has any detection from 2013 on, for every month from 2000-11 to the end of the analysis, zeros included

There is deliberately no year input: the reconstruction of 2003-2012 must not depend on a year the model never saw.

In [ ]:
MONTHS_BM = [str(p) for p in pd.period_range("2000-11", f"{LAST_FULL_YEAR}-12", freq="M")]

def block_month_table(scope):
    modis, viirs = select("modis", scope), select("viirs", scope)
    def counts(d):
        d = d.drop_duplicates(["cy", "cx", "date"])
        return d.groupby(["by", "bx", "ym"]).size()
    m_all = counts(modis)
    m_terra = counts(modis[modis.sat.str.contains("Terra", case=False)])
    m_aqua = counts(modis[modis.sat.str.contains("Aqua", case=False)])
    v = counts(viirs[viirs.date >= "2013-01-01"])
    in_range = m_all.index.get_level_values("ym") >= "2013-01"
    blocks = pd.concat([m_all[in_range].reset_index()[["by", "bx"]], v.reset_index()[["by", "bx"]]]).drop_duplicates().reset_index(drop=True)
    bm = blocks.merge(pd.DataFrame({"ym": MONTHS_BM}), how="cross")
    idx = pd.MultiIndex.from_frame(bm[["by", "bx", "ym"]])
    bm["m_all"] = m_all.reindex(idx).fillna(0).to_numpy()
    bm["m_terra"] = m_terra.reindex(idx).fillna(0).to_numpy()
    bm["m_aqua"] = m_aqua.reindex(idx).fillna(0).to_numpy()
    bm["v"] = v.reindex(idx).fillna(0).to_numpy()
    bm.loc[bm.ym < "2013-01", "v"] = np.nan                    # no VIIRS truth before 2013
    bm["year"], bm["month"] = bm.ym.str[:4].astype(int), bm.ym.str[5:7].astype(int)
    bm["blat"], bm["blon"] = (bm.by + 0.5) * BLOCK, (bm.bx + 0.5) * BLOCK
    bm["msin"], bm["mcos"] = np.sin(2 * np.pi * bm.month / 12), np.cos(2 * np.pi * bm.month / 12)
    for c in ("all", "terra", "aqua"):
        bm["lm_" + c] = np.log1p(bm["m_" + c])
    return bm

BM = {scope: block_month_table(scope) for scope in ("type0", "all")}
summary = {}
for scope, bm in BM.items():
    known = bm[bm.v.notna()]
    summary[scope] = {"blocks": bm[["by", "bx"]].drop_duplicates().shape[0], "rows": len(bm),
                      "VIIRS cell-days 2013+": known.v.sum(), "MODIS cell-days 2013+": bm[bm.year >= 2013].m_all.sum(),
                      "share of VIIRS cell-days in blocks with no MODIS that month (%)": 100 * known[known.m_all == 0].v.sum() / known.v.sum()}
display(keep("G_block_month_summary", pd.DataFrame(summary).round(1)))
print("minutes so far:", round((time.time() - T0) / 60, 1))

## H. The four translation models

Each block below is one model. All of them take the training rows of one scenario and return a `predict` function.
All four are calibrated so that their total over the training rows equals the real VIIRS total (the factor `smear`), because the models for log-counts otherwise tend to under-predict totals.

1. **Straight log-log curve:** `log(1+VIIRS) = a + b * log(1+MODIS)`. One line for the whole country.
2. **Curved fit:** the same with a squared term, so the curve can bend at high activity (this was the best total model in the earlier work).
3. **Gradient boosting (machine learning):** many small decision trees, trained with a Poisson loss because the target is a count.
4. **Neural network (deep learning):** two hidden layers of 64 and 32 units, trained on `log(1+VIIRS)`, stopping itself when its internal validation score stops improving.

Only the cell-level and block-level inputs listed in section G are used. All inputs come from MODIS, so the same models can run in 2000-2012.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

FEATURES = ["lm_all", "lm_terra", "lm_aqua", "blat", "blon", "msin", "mcos"]

# ---------------- model 1: straight log-log curve ----------------
def fit_straight(train):
    x = train["lm_all"].to_numpy()
    y = np.log1p(train["v"].to_numpy())
    b, a = np.polyfit(x, y, 1)                                           # y = a + b x
    smear = train["v"].sum() / np.expm1(a + b * x).sum()                 # makes the training total match
    def predict(df):
        return np.clip(np.expm1(a + b * df["lm_all"].to_numpy()), 0, None) * smear
    return predict

# ---------------- model 2: curved (quadratic) log-log fit ----------------
def fit_curved(train):
    x = train["lm_all"].to_numpy()
    y = np.log1p(train["v"].to_numpy())
    c2, c1, c0 = np.polyfit(x, y, 2)                                     # y = c0 + c1 x + c2 x^2
    smear = train["v"].sum() / np.expm1(c0 + c1 * x + c2 * x ** 2).sum()
    def predict(df):
        z = df["lm_all"].to_numpy()
        return np.clip(np.expm1(c0 + c1 * z + c2 * z ** 2), 0, None) * smear
    return predict

# ---------------- model 3: gradient boosting ----------------
def fit_gradient_boosting(train):
    model = HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=50, l2_regularization=1.0, random_state=0)
    model.fit(train[FEATURES], train["v"])
    def predict(df):
        return model.predict(df[FEATURES])
    return predict

# ---------------- model 4: neural network ----------------
def fit_neural_network(train, seed=0):
    scaler = StandardScaler().fit(train[FEATURES])
    model = MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=200, random_state=seed)
    model.fit(scaler.transform(train[FEATURES]), np.log1p(train["v"]))
    fitted = np.clip(np.expm1(model.predict(scaler.transform(train[FEATURES]))), 0, None)
    smear = train["v"].sum() / fitted.sum()
    def predict(df):
        return np.clip(np.expm1(model.predict(scaler.transform(df[FEATURES]))), 0, None) * smear
    return predict

MODEL_FITS = {"straight log-log": fit_straight, "curved log-quadratic": fit_curved,
              "gradient boosting": fit_gradient_boosting, "neural network": fit_neural_network}

# ---------------- the score ----------------
def score(df, predicted, months):
    mask = df.ym.isin(months).to_numpy()                       # keep only the months being scored
    d = df[mask].copy()
    d["pred"] = np.asarray(predicted)[mask]
    monthly = d.groupby("ym")[["v", "pred"]].sum()
    monthly = monthly[monthly.v > 0]
    out = {"median monthly error (%)": 100 * np.median(np.abs(monthly.pred - monthly.v) / monthly.v),
           "total bias (%)": 100 * (monthly.pred.sum() / monthly.v.sum() - 1),
           "block MAE (cell-days)": float(np.mean(np.abs(d.pred - d.v))),
           "block RMSE (cell-days)": float(np.sqrt(np.mean((d.pred - d.v) ** 2))),
           "block Spearman": d["v"].corr(d["pred"], method="spearman")}
    for name, mask in (("north", d.blat >= NORTH_LAT), ("south", d.blat < NORTH_LAT)):
        out[f"bias {name} (%)"] = 100 * (d.pred[mask].sum() / d.v[mask].sum() - 1) if d.v[mask].sum() > 0 else np.nan
    return out
print("models defined")

## I. Comparing the data decisions

The same four models are trained in four scenarios and scored on the same hidden test: 2013-2015 (VIIRS hidden), excluding outage months. The scenarios change one decision at a time:

- **S1** type 0 only, outage months removed from training, trained on 2016-2021 (years with stable MODIS overpass times; the choice we want to defend)
- **S2** the same, but trained on 2016-2025 (adds the drift years to the training data)
- **S3** like S2, but with the outage months kept in training
- **S4** all detection types, outage months removed, trained on 2016-2021

Read S1 against S2 for the drift years, S2 against S3 for the outage months, S1 against S4 for the type choice.
One caution: S4 is scored against all-type VIIRS counts, a different target from the other three, so only its relative errors (the percentages) can be compared, not its block errors in cell-days.

In [ ]:
def months_in(y0, y1):
    return [m for m in MONTHS_BM if y0 <= int(m[:4]) <= y1]

TEST_MONTHS = [m for m in months_in(*HIDDEN_YEARS) if m not in OUTAGE_ALL]
SCENARIOS = {
    "S1 type0 | outages removed | train 2016-2021": dict(scope="type0", drop_outages=True, train=(2016, 2021)),
    "S2 type0 | outages removed | train 2016-2025": dict(scope="type0", drop_outages=True, train=(2016, 2025)),
    "S3 type0 | outages KEPT | train 2016-2025": dict(scope="type0", drop_outages=False, train=(2016, 2025)),
    "S4 ALL types | outages removed | train 2016-2021": dict(scope="all", drop_outages=True, train=(2016, 2021)),
}
print("test months:", len(TEST_MONTHS), "| hidden years", HIDDEN_YEARS)

rows, TRAINED = [], {}
for sname, cfg in SCENARIOS.items():
    bm = BM[cfg["scope"]]
    train_months = months_in(*cfg["train"])
    if cfg["drop_outages"]:
        train_months = [m for m in train_months if m not in OUTAGE_ALL]
    train = bm[bm.ym.isin(train_months) & bm.v.notna()]
    test = bm[bm.ym.isin(TEST_MONTHS)]
    for mname, fit in MODEL_FITS.items():
        predict = fit(train)
        TRAINED[(sname, mname)] = predict
        res = score(test, predict(test), TEST_MONTHS)
        res.update({"scenario": sname, "model": mname, "train rows": len(train)})
        rows.append(res)
    print(sname, "done | minutes so far:", round((time.time() - T0) / 60, 1))
SCEN = pd.DataFrame(rows)
cols = ["scenario", "model", "train rows", "median monthly error (%)", "total bias (%)", "bias north (%)", "bias south (%)",
        "block MAE (cell-days)", "block RMSE (cell-days)", "block Spearman"]
display(keep("I_scenarios", SCEN[cols].round(2)))

In [ ]:
short = {s: s.split(" ")[0] for s in SCENARIOS}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, metric in zip(axes, ["median monthly error (%)", "total bias (%)"]):
    pivot = SCEN.pivot(index="scenario", columns="model", values=metric).reindex(list(SCENARIOS))
    pivot.index = [short[s] for s in pivot.index]
    pivot.plot(kind="bar", ax=ax, width=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"Hidden test {HIDDEN_YEARS[0]}-{HIDDEN_YEARS[1]}: {metric}"); ax.set_xlabel("scenario (S4 uses a different truth)")
    ax.tick_params(axis="x", rotation=0)
savefig("I_scenario_comparison")

# predicted against actual monthly totals in the test months, scenario S1
s1 = list(SCENARIOS)[0]
bm = BM["type0"]
test = bm[bm.ym.isin(TEST_MONTHS)]
fig, ax = plt.subplots(figsize=(14, 4.5))
actual = test.groupby("ym").v.sum()
ax.plot(pd.to_datetime(actual.index + "-01"), actual.values, color="k", lw=2.5, label="real VIIRS")
for mname in MODEL_FITS:
    pred = TRAINED[(s1, mname)](test)
    monthly = pd.Series(pred, index=test.index).groupby(test.ym).sum()
    ax.plot(pd.to_datetime(monthly.index + "-01"), monthly.values, marker=".", label=mname)
ax.set_title(f"{s1}: monthly VIIRS cell-days on the hidden test"); ax.set_ylabel("country total, cell-days"); ax.legend()
savefig("I_test_monthly_totals")

In [ ]:
# the same models (scenario S1, trained 2016-2021) applied to every year with real VIIRS: where does the bias drift?
valid_months = [m for m in MONTHS_BM if int(m[:4]) >= 2013 and m not in OUTAGE_ALL]
all_years = BM["type0"][BM["type0"].ym.isin(valid_months)]
yearly = {}
for mname in MODEL_FITS:
    pred = pd.Series(TRAINED[(s1, mname)](all_years), index=all_years.index)
    by_year = pd.DataFrame({"v": all_years.v, "pred": pred, "year": all_years.year}).groupby("year")[["v", "pred"]].sum()
    yearly[mname] = 100 * (by_year.pred / by_year.v - 1)
YEARLY = pd.DataFrame(yearly)
display(keep("I_yearly_bias_S1", YEARLY.round(1)))

fig, ax = plt.subplots(figsize=(12, 4.5))
for mname in YEARLY:
    ax.plot(YEARLY.index, YEARLY[mname], marker="o", label=mname)
ax.axhline(0, color="k", lw=0.8)
ax.axvspan(2015.5, 2021.5, color="tab:green", alpha=0.08, label="training years (in-sample)")
ax.axvspan(TEST_YEARS[0] - 0.5, TEST_YEARS[1] + 0.5, color="grey", alpha=0.15, label="drift years")
ax.set_title("Yearly bias of the S1 models, outage months removed (0 = unbiased)"); ax.set_xlabel("year"); ax.set_ylabel("bias (%)"); ax.legend(fontsize=8)
savefig("I_yearly_bias")

## J. How stable is the neural network?

A neural network starts from random weights, so two runs give different answers. Before comparing it with anything, we need to know how large that wobble is.
Same data (scenario S1), five different random seeds.

In [ ]:
bm = BM["type0"]
cfg = SCENARIOS[s1]
train_months = [m for m in months_in(*cfg["train"]) if m not in OUTAGE_ALL]
train = bm[bm.ym.isin(train_months) & bm.v.notna()]
test = bm[bm.ym.isin(TEST_MONTHS)]
seed_rows = []
for seed in range(5):
    predict = fit_neural_network(train, seed=seed)
    res = score(test, predict(test), TEST_MONTHS)
    seed_rows.append({"seed": seed, "median monthly error (%)": res["median monthly error (%)"],
                      "total bias (%)": res["total bias (%)"], "block MAE (cell-days)": res["block MAE (cell-days)"]})
SEEDS = pd.DataFrame(seed_rows).set_index("seed")
display(keep("J_nn_seeds", SEEDS.round(2)))
print("mean:", SEEDS.mean().round(2).to_dict())
print("std: ", SEEDS.std().round(2).to_dict())
gb_row = SCEN[(SCEN.scenario == s1) & (SCEN.model == "gradient boosting")].iloc[0]
print("gradient boosting, same scenario (one fit, deterministic):", round(gb_row["median monthly error (%)"], 2), "%  /  bias", round(gb_row["total bias (%)"], 2), "%")

## K. Preview of the reconstruction 2000-2012, and the one-satellite problem

The scenario S1 models are applied to the MODIS-only months before VIIRS. This is a preview, not a validated product: nothing before 2013 can be checked against VIIRS.
Per-month averages are used so that partial years (2000 starts in November, 2012 stops in June) are not penalised.
What to look for: a step in the predicted series when Aqua starts (the model never saw a month with no Aqua), and whether the predicted level before 2013 joins the real VIIRS level after it.

In [ ]:
aqua_first = D["modis"][D["modis"].sat.str.contains("Aqua", case=False)].date.min()
terra_first = D["modis"][D["modis"].sat.str.contains("Terra", case=False)].date.min()
print("first Terra row:", terra_first.date(), "| first Aqua row:", aqua_first.date())

bm = BM["type0"]
pre = bm[(bm.ym < "2012-07") & (~bm.ym.isin(OUTAGE_ALL))]
real = bm[(bm.ym >= "2013-01") & (~bm.ym.isin(OUTAGE_ALL))]
frames = {}
for mname in ("curved log-quadratic", "gradient boosting", "neural network"):
    pred = pd.Series(TRAINED[(s1, mname)](pre), index=pre.index)
    monthly = pred.groupby(pre.ym).sum()
    frames[f"{mname} (predicted VIIRS)"] = monthly.groupby(lambda m: int(m[:4])).mean()
frames["MODIS cell-days (input)"] = pre.groupby("ym").m_all.sum().groupby(lambda m: int(m[:4])).mean()
frames["real VIIRS"] = real.groupby("ym").v.sum().groupby(lambda m: int(m[:4])).mean()
RECON = pd.DataFrame(frames)
display(keep("K_reconstruction_preview_mean_per_month", RECON.round(0)))

fig, ax = plt.subplots(figsize=(13, 4.8))
for col in RECON.columns:
    style = {"lw": 3, "color": "k"} if col == "real VIIRS" else ({"ls": "--"} if col.startswith("MODIS") else {})
    ax.plot(RECON.index, RECON[col], marker="o", label=col, **style)
ax.axvspan(2000, aqua_first.year + aqua_first.dayofyear / 365, color="tab:orange", alpha=0.15, label="Terra only")
ax.set_title("Average fire cell-days per month, by year (type 0, outage months removed)"); ax.set_xlabel("year"); ax.set_ylabel("country total per month")
ax.legend(fontsize=8)
savefig("K_reconstruction_preview")

## Save the tables

All tables are written as CSV next to the figures, so they can be pasted or shared.

In [ ]:
tables_dir = FIGDIR / "tables"
tables_dir.mkdir(exist_ok=True)
for name, table in RESULTS.items():
    table.to_csv(tables_dir / f"{name}.csv")
print("saved", len(RESULTS), "tables to", tables_dir.resolve())
print("saved figures to", FIGDIR.resolve())
print("total time:", round((time.time() - T0) / 60, 1), "minutes")